# HuBERT Gating Network Notebook

This notebook builds and validates a learnable temporal gating network for PD vs HC using precomputed HuBERT embeddings.

Execution style: run cells sequentially from top to bottom.

## 1. Environment Setup and Reproducibility

In [1]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "torch",
    "torchaudio",
    "transformers",
    "datasets",
    "evaluate",
    "scikit-learn",
    "matplotlib",
    "pandas",
    "numpy",
    "soundfile",
], check=True)

print("Dependencies installed.")

Dependencies installed.


In [2]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("torch:", torch.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)

Device: cpu
torch: 2.10.0+cu128
numpy: 2.4.2
pandas: 3.0.1


## 2. Project Configuration and Hyperparameters

In [3]:
CFG = {
    "batch_size": 64,
    "smoke_test_samples": 300,
    "contrastive_temperature": 0.07,
}

ROOT = Path.cwd().resolve()
META_PATH = ROOT / "metadata.csv"
EMB_ROOT = ROOT / "embeddings"
OUT_DIR = ROOT / "results" / "gating_network"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("META_PATH:", META_PATH)
print("EMB_ROOT:", EMB_ROOT)
print("OUT_DIR:", OUT_DIR)
print(CFG)

ROOT: /home/bs00956/Desktop/Personal/PD/Pipeline-Implementation/HuBERT
META_PATH: /home/bs00956/Desktop/Personal/PD/Pipeline-Implementation/HuBERT/metadata.csv
EMB_ROOT: /home/bs00956/Desktop/Personal/PD/Pipeline-Implementation/HuBERT/embeddings
OUT_DIR: /home/bs00956/Desktop/Personal/PD/Pipeline-Implementation/HuBERT/results/gating_network
{'batch_size': 64, 'smoke_test_samples': 300, 'contrastive_temperature': 0.07}


## 3. Load HuBERT Embedding Metadata

In [4]:
meta = pd.read_csv(META_PATH)
meta = meta[meta["segment"].isin(["early", "middle", "late"])].copy()

if "embedding_path" not in meta.columns:
    raise ValueError("metadata.csv must contain 'embedding_path' column.")

# Keep only mean-pooled embeddings and full triplets per subject.
meta = meta[~meta["embedding_path"].astype(str).str.endswith("_seq.npy")].copy()
counts = meta.groupby(["original_stem", "class"])["segment"].nunique().reset_index()
full_subjects = counts[counts["segment"] == 3][["original_stem", "class"]]
full_meta = meta.merge(full_subjects, on=["original_stem", "class"], how="inner")

subject_rows = full_meta[["original_stem", "class"]].drop_duplicates()
if len(subject_rows) > CFG["smoke_test_samples"]:
    subject_rows = subject_rows.sample(n=CFG["smoke_test_samples"], random_state=SEED)
full_meta = full_meta.merge(subject_rows, on=["original_stem", "class"], how="inner")

print("Subjects selected:", subject_rows.shape[0])
print("Rows selected:", len(full_meta))
print(full_meta[["class", "segment"]].value_counts().sort_index())

Subjects selected: 300
Rows selected: 900
class  segment
HC     early      217
       late       217
       middle     217
PD     early       83
       late        83
       middle      83
Name: count, dtype: int64


## 4. Build Embedding Dataset and Data Collator

In [5]:
def to_abs_path(p: str) -> Path:
    path = Path(p)
    if path.is_absolute():
        return path
    return ROOT / path

records = []
for (stem, cls), grp in full_meta.groupby(["original_stem", "class"]):
    seg_map = {r.segment: r for r in grp.itertuples(index=False)}
    if not all(k in seg_map for k in ["early", "middle", "late"]):
        continue

    try:
        p_early = to_abs_path(seg_map["early"].embedding_path)
        p_middle = to_abs_path(seg_map["middle"].embedding_path)
        p_late = to_abs_path(seg_map["late"].embedding_path)

        row = {
            "original_stem": stem,
            "label": 1 if cls == "PD" else 0,
            "emb_early": np.load(p_early).astype(np.float32),
            "emb_middle": np.load(p_middle).astype(np.float32),
            "emb_late": np.load(p_late).astype(np.float32),
        }
        records.append(row)
    except (FileNotFoundError, OSError, ValueError):
        continue

data = Dataset.from_list(records)
print("Prepared subject-level embedding dataset:", len(data))
print("Label counts:", pd.Series(data["label"]).value_counts().to_dict())

class SubjectEmbeddingCollator:
    def __call__(self, examples):
        e = torch.tensor(np.stack([ex["emb_early"] for ex in examples]), dtype=torch.float32)
        m = torch.tensor(np.stack([ex["emb_middle"] for ex in examples]), dtype=torch.float32)
        l = torch.tensor(np.stack([ex["emb_late"] for ex in examples]), dtype=torch.float32)
        y = torch.tensor([ex["label"] for ex in examples], dtype=torch.long)
        return {
            "f_early": e,
            "f_middle": m,
            "f_late": l,
            "labels": y,
        }

collator = SubjectEmbeddingCollator()
print("Embedding collator ready.")

Prepared subject-level embedding dataset: 300
Label counts: {0: 217, 1: 83}
Embedding collator ready.


## 5. Implement the Gating Network Module

In [6]:
class TemporalGating(nn.Module):
    def __init__(self, feature_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(feature_dim * 3, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 3),
        )

    def forward(self, f_early, f_middle, f_late):
        x = torch.cat([f_early, f_middle, f_late], dim=-1)
        logits = self.mlp(x)
        gates = torch.softmax(logits, dim=-1)
        f_gated = (
            gates[:, 0:1] * f_early
            + gates[:, 1:2] * f_middle
            + gates[:, 2:3] * f_late
        )
        return f_gated, gates

# Unit sanity check on actual embedding dimension from dataset
D = len(data[0]["emb_early"])
B = 4
gate_mod = TemporalGating(feature_dim=D)
fe = torch.randn(B, D)
fm = torch.randn(B, D)
fl = torch.randn(B, D)
fused, gs = gate_mod(fe, fm, fl)
assert fused.shape == (B, D)
assert gs.shape == (B, 3)
assert torch.allclose(gs.sum(dim=-1), torch.ones(B), atol=1e-5)
print("TemporalGating sanity check passed for D=", D)

TemporalGating sanity check passed for D= 768


## 6. Temporal Contrast and Projection (Using Existing HuBERT F_early/F_middle/F_late Embeddings)

In [7]:
from torch.utils.data import DataLoader

class TemporalContrast(nn.Module):
    def forward(self, f_early, f_late):
        return f_late - f_early

class ProjectionHead(nn.Module):
    def __init__(self, input_dim: int, projection_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.ReLU(),
            nn.Linear(input_dim, projection_dim),
        )

    def forward(self, x):
        z = self.net(x)
        return F.normalize(z, dim=-1)

class TemporalContrastEmbeddingEncoder(nn.Module):
    """Uses precomputed segment embeddings directly (no backbone re-encoding)."""

    def __init__(self, feature_dim: int):
        super().__init__()
        self.gating = TemporalGating(feature_dim)
        self.temporal_contrast = TemporalContrast()
        self.projection = ProjectionHead(input_dim=feature_dim * 2, projection_dim=128)

    def forward(self, batch, return_outputs=False):
        f_e = batch["f_early"]
        f_m = batch["f_middle"]
        f_l = batch["f_late"]

        f_gated, gates = self.gating(f_e, f_m, f_l)
        delta_f = self.temporal_contrast(f_e, f_l)
        contrastive_input = torch.cat([f_gated, delta_f], dim=-1)
        z = self.projection(contrastive_input)

        if return_outputs:
            return {
                "gates": gates,
                "f_gated": f_gated,
                "delta_f": delta_f,
                "contrastive_input": contrastive_input,
                "projection": z,
            }
        return z

# quick shape check
sample_loader = DataLoader(data.select(range(min(4, len(data)))), batch_size=2, collate_fn=collator)
sample_batch = next(iter(sample_loader))
sample_batch = {k: v.to(DEVICE) for k, v in sample_batch.items()}
encoder_check = TemporalContrastEmbeddingEncoder(feature_dim=D).to(DEVICE)
with torch.no_grad():
    out = encoder_check(sample_batch, return_outputs=True)
print(
    "Gates shape:", tuple(out["gates"].shape),
    "| DeltaF shape:", tuple(out["delta_f"].shape),
    "| Projection shape:", tuple(out["projection"].shape),
)

Gates shape: (2, 3) | DeltaF shape: (2, 768) | Projection shape: (2, 128)


## 7. Contrastive Learning (Positive/Negative Pairs) + Optimizer

In [8]:
class SupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor):
        if embeddings.ndim != 2:
            raise ValueError("embeddings must be [batch, dim].")
        if labels.ndim != 1:
            raise ValueError("labels must be [batch].")

        z = F.normalize(embeddings, dim=-1)
        sim = torch.matmul(z, z.T) / self.temperature

        labels = labels.view(-1, 1)
        pos_mask = torch.eq(labels, labels.T).float()

        logits_mask = torch.ones_like(pos_mask) - torch.eye(pos_mask.size(0), device=pos_mask.device)
        pos_mask = pos_mask * logits_mask

        exp_sim = torch.exp(sim) * logits_mask
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-12)

        pos_count = pos_mask.sum(dim=1)
        mean_log_prob_pos = (pos_mask * log_prob).sum(dim=1) / (pos_count + 1e-12)

        valid = pos_count > 0
        if valid.any():
            return -mean_log_prob_pos[valid].mean()
        return torch.tensor(0.0, device=embeddings.device)

contrastive_criterion = SupervisedContrastiveLoss(
    temperature=CFG["contrastive_temperature"]
)

# Single-batch sanity check for contrastive objective
sample_loader = DataLoader(data.select(range(min(16, len(data)))), batch_size=8, collate_fn=collator)
sample_batch = next(iter(sample_loader))
sample_batch = {k: v.to(DEVICE) for k, v in sample_batch.items()}

contrastive_encoder = TemporalContrastEmbeddingEncoder(feature_dim=D).to(DEVICE)
contrastive_encoder.eval()
with torch.no_grad():
    out = contrastive_encoder(sample_batch, return_outputs=True)
    z = out["projection"]
    c_loss = contrastive_criterion(z, sample_batch["labels"])

print("Contrastive projection shape:", tuple(z.shape))
print("Contrastive loss (sanity):", float(c_loss.item()))
print("Mean gate weights (e,m,l):", out["gates"].mean(dim=0).cpu().numpy().round(4))

Contrastive projection shape: (8, 128)
Contrastive loss (sanity): 2.6807634830474854
Mean gate weights (e,m,l): [0.3029 0.3336 0.3636]


## 8. Feature Fusion and Classification Head (Deferred)

This notebook stage intentionally stops at **Contrastive Learning**.
The downstream fusion and classifier blocks will be added in the next phase.

In [9]:
print("Deferred in this stage: Feature Fusion + Classification training loop.")
print("Current notebook scope ends at contrastive embedding learning setup.")

Deferred in this stage: Feature Fusion + Classification training loop.
Current notebook scope ends at contrastive embedding learning setup.


## 9. Metrics and Gating Visualization (Deferred)

In [10]:
print("Deferred in this stage: validation metrics plots.")

Deferred in this stage: validation metrics plots.


## 10. Save, Reload, and Inference (Deferred)

In [11]:
print("Deferred in this stage: checkpointing and inference.")

Deferred in this stage: checkpointing and inference.


## 11. Lightweight Forward-Pass Diagnostics (No Training)

This diagnostic validates three things without training and is CPU-friendly:
1. Gate weights (`w_e, w_m, w_l`) and gate entropy (uniform vs differentiated).
2. `F_gated` class structure via within-class vs cross-class cosine similarity.
3. `DeltaF` (`F_late - F_early`) magnitude difference between PD and HC.

In [ ]:
from torch.utils.data import DataLoader
from sklearn.metrics.pairwise import cosine_similarity

@torch.no_grad()
def run_lightweight_diagnostics(dataset, collate_fn, feature_dim, device, batch_size=16):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    model = TemporalGating(feature_dim=feature_dim).to(device)
    model.eval()

    all_fg, all_df, all_gates, all_labels = [], [], [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        f_gated, gates = model(batch["f_early"], batch["f_middle"], batch["f_late"])
        delta_f = batch["f_late"] - batch["f_early"]

        all_fg.append(f_gated.cpu())
        all_df.append(delta_f.cpu())
        all_gates.append(gates.cpu())
        all_labels.append(batch["labels"].cpu())

    f_gated = torch.cat(all_fg, dim=0)
    delta_f = torch.cat(all_df, dim=0)
    gates = torch.cat(all_gates, dim=0)
    labels = torch.cat(all_labels, dim=0).numpy()

    gate_mean = gates.mean(dim=0).numpy()
    gate_std = gates.std(dim=0).numpy()
    gate_entropy = float((-(gates * (gates + 1e-12).log()).sum(dim=1)).mean().item())
    uniform_entropy = float(np.log(3.0))

    fg_norm = F.normalize(f_gated, dim=-1).numpy()
    pd_mask = labels == 1
    hc_mask = labels == 0

    def within_mean(mask):
        idx = np.where(mask)[0]
        if len(idx) < 2:
            return float("nan")
        sim = cosine_similarity(fg_norm[idx], fg_norm[idx])
        upper = sim[np.triu_indices_from(sim, k=1)]
        return float(upper.mean()) if upper.size else float("nan")

    def cross_mean(mask_a, mask_b):
        if mask_a.sum() == 0 or mask_b.sum() == 0:
            return float("nan")
        sim = cosine_similarity(fg_norm[mask_a], fg_norm[mask_b])
        return float(sim.mean())

    within_pd = within_mean(pd_mask)
    within_hc = within_mean(hc_mask)
    cross_ph = cross_mean(pd_mask, hc_mask)
    cross_hp = cross_mean(hc_mask, pd_mask)
    cross_avg = float(np.nanmean([cross_ph, cross_hp]))

    delta_norm = delta_f.norm(dim=1).numpy()
    pd_delta = float(delta_norm[pd_mask].mean()) if pd_mask.any() else float("nan")
    hc_delta = float(delta_norm[hc_mask].mean()) if hc_mask.any() else float("nan")

    pd_gate = gates[pd_mask].mean(dim=0).numpy() if pd_mask.any() else np.array([np.nan, np.nan, np.nan])
    hc_gate = gates[hc_mask].mean(dim=0).numpy() if hc_mask.any() else np.array([np.nan, np.nan, np.nan])

    print(f"Subjects used: {len(dataset)}")
    print(f"Gate mean (e,m,l): {np.round(gate_mean, 4)}")
    print(f"Gate std  (e,m,l): {np.round(gate_std, 4)}")
    print(f"Gate entropy: {gate_entropy:.4f} / uniform={uniform_entropy:.4f}")
    print()
    print("Class-wise gate means:")
    print(f"  PD: {np.round(pd_gate, 4)}")
    print(f"  HC: {np.round(hc_gate, 4)}")
    print(f"  late weight gap (PD - HC): {pd_gate[2] - hc_gate[2]:.4f}")
    print()
    print("F_gated cosine similarity:")
    print(f"  within PD : {within_pd:.4f}")
    print(f"  within HC : {within_hc:.4f}")
    print(f"  cross PD/HC: {cross_avg:.4f}")
    print()
    print("DeltaF norm:")
    print(f"  PD mean: {pd_delta:.4f}")
    print(f"  HC mean: {hc_delta:.4f}")
    print(f"  PD - HC : {pd_delta - hc_delta:.4f}")
    print()
    print("Quick read:")
    if gate_entropy > uniform_entropy - 0.05:
        print("- Gates are close to uniform (expected before training).")
    else:
        print("- Gates are differentiated even without training.")

    if not np.isnan(cross_avg) and within_pd > cross_avg and within_hc > cross_avg:
        print("- F_gated shows within-class structure above cross-class similarity.")
    else:
        print("- F_gated does not yet show clear class separation.")

    if not np.isnan(pd_delta) and not np.isnan(hc_delta) and pd_delta > hc_delta:
        print("- DeltaF magnitude follows expected PD > HC direction.")
    else:
        print("- DeltaF magnitude gap is not yet in the expected direction.")

run_lightweight_diagnostics(data, collator, D, DEVICE, batch_size=16)